# Response Ranking for Persian Social-Commerce Conversations
## Professor-Ready Experiment Notebook — PerSHOP Benchmark

**Student:** Bahar Shafieian  
**Supervisors:** Prof. Ciro Russo and Prof. Mario Guarracino  
**Programme:** BSc Economics with Data Science, University of Cassino and Southern Lazio  
**Notebook version:** 24 July 2026 — updated with completed Dorna2 fair-condition results

---

### Purpose

This notebook consolidates the completed PerSHOP experiments into one ordered and reviewable workflow. It was reconstructed from the original experimental file (`dorna2_pershop.py`) and the later clean notebook containing SFT, RAG-ICL, Merge, and fair-condition Dorna2 evaluation.

The notebook deliberately **does not recreate the TF-IDF, BM25, or standalone BGE-m3 baseline implementations**, because those were developed separately in PyCharm. Their recorded results remain in the final comparison table and are labelled as external implementations.

### Use

- Completed results are written immediately after each method.
- Heavy GPU execution is disabled by default through `RUN_HEAVY_EXPERIMENTS = False`.
- Detailed predictions are saved to JSONL and summaries to CSV.
- Expensive evaluations are checkpointed and resumable.
- Gemma, Llama, Dorna2, SFT, and Merge sections have explicit fresh-runtime boundaries.
- Blank result values are never guessed; they mean the downloaded summary was not embedded in the supplied source files.


## Ordered experiment map

1. Environment and reproducibility  
2. Shared parsing, metrics, logging, and checkpoint utilities  
3. Dataset sanity checks  
4. Original single-pass Llama and Gemma baselines  
5. Heavy-prompt collapse diagnostic  
6. Zero-shot position-bias diagnostic with PSC  
7. Lean systematic prompt with PSC  
8. Chain-of-Prompting with PSC  
9. Curriculum SFT and SFT-only evaluation  
10. RAG-ICL  
11. Merge: RAG-augmented SFT and confirmation runs  
12. Original Dorna2 exploratory protocols  
13. Dorna2 fair-condition evaluation  
14. Unified result registry and error inspection


# 1. Environment and reproducibility


In [ ]:
# Run once in a fresh Colab runtime.
!pip install -q -U \
    bitsandbytes transformers accelerate peft trl datasets \
    sentence-transformers huggingface_hub pandas numpy

# Colab may require: Runtime -> Restart session


In [ ]:
import os
import gc
import re
import csv
import json
import math
import time
import random
import platform
import subprocess
import sys
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Prevent accidental execution of multi-hour experiments.
RUN_HEAVY_EXPERIMENTS = False

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/PerSHOP_Thesis")
else:
    PROJECT_ROOT = Path("/content/PerSHOP_Thesis")

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
ADAPTER_DIR = PROJECT_ROOT / "adapters"
RUNTIME_DIR = PROJECT_ROOT / "runtime"

for folder in [DATA_DIR, RESULTS_DIR, CHECKPOINT_DIR, ADAPTER_DIR, RUNTIME_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

TEST_FILES = {
    "random": "pershop_random_test.jsonl",
    "domain": "pershop_domain_test.jsonl",
    "domain_lexical": "pershop_domain_lexical_test.jsonl",
}
TRAIN_FILES = {
    "random": "pershop_random_train.jsonl",
    "domain": "pershop_domain_train.jsonl",
    "domain_lexical": "pershop_domain_lexical_train.jsonl",
}

print("Project root:", PROJECT_ROOT)
print("Current data files:", sorted(path.name for path in DATA_DIR.glob("*")))


In [ ]:
requirements_path = RUNTIME_DIR / "requirements_runtime.txt"
with requirements_path.open("w", encoding="utf-8") as output:
    subprocess.run(
        [sys.executable, "-m", "pip", "freeze"],
        stdout=output,
        check=False,
    )

runtime_metadata = {
    "seed": SEED,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}
with (RUNTIME_DIR / "runtime_metadata.json").open("w", encoding="utf-8") as output:
    json.dump(runtime_metadata, output, ensure_ascii=False, indent=2)

print("Saved runtime metadata and package versions.")


# 2. Shared benchmark, parsing, metrics, and output utilities

Each instance contains five seller responses: one gold response and four negatives. The three evaluation strategies increase in difficulty from `random` to `domain` to `domain_lexical`.


In [ ]:
def load_jsonl(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    with path.open(encoding="utf-8") as source:
        return [json.loads(line) for line in source if line.strip()]

def append_jsonl(path, row):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as output:
        output.write(json.dumps(row, ensure_ascii=False) + "\n")

def extract_query(instance):
    for key in ["query", "customer_message", "customer_query", "message", "question", "text"]:
        if instance.get(key) is not None:
            return str(instance[key])
    raise KeyError("No query field found.")

def extract_candidates(instance):
    for key in ["candidates", "responses", "candidate_responses", "answers"]:
        if instance.get(key) is not None:
            return list(instance[key])
    raise KeyError("No candidates field found.")

def candidate_text(candidate):
    if isinstance(candidate, dict):
        for key in ["text", "response", "answer", "candidate_text", "seller_response"]:
            if candidate.get(key) is not None:
                return str(candidate[key])
    return str(candidate)

def get_gold_idx(instance):
    candidates = extract_candidates(instance)
    for index, candidate in enumerate(candidates):
        if isinstance(candidate, dict):
            value = candidate.get("label", candidate.get("is_gold", candidate.get("gold")))
            if value is True or str(value).strip().lower() in {
                "1", "true", "yes", "gold", "correct"
            }:
                return index
    for key in ["gold_idx", "gold_index", "correct_idx", "correct_index"]:
        if key in instance:
            return int(instance[key])
    raise ValueError("Could not identify the gold response.")

def normalize_digits(text):
    return str(text).translate(
        str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")
    )

def parse_ranking_bracket(raw_text, n=5):
    text = normalize_digits(raw_text).strip()

    chain_pattern = re.compile(
        r"(?:\[\d+\]\s*>\s*){" + str(n - 1) + r"}\[\d+\]"
    )
    chains = chain_pattern.findall(text)
    if chains:
        numbers = [int(value) for value in re.findall(r"\d+", chains[-1])]
        if len(numbers) == n and sorted(numbers) == list(range(1, n + 1)):
            return [value - 1 for value in numbers], True, "bracket_chain"

    for array_text in re.findall(r"\[[^\]]+\]", text):
        numbers = [int(value) for value in re.findall(r"\d+", array_text)]
        if len(numbers) == n and sorted(numbers) == list(range(1, n + 1)):
            return [value - 1 for value in numbers], True, "array"

    bracket_numbers = [int(value) for value in re.findall(r"\[(\d+)\]", text)]
    if len(bracket_numbers) >= n:
        tail = bracket_numbers[-n:]
        if sorted(tail) == list(range(1, n + 1)):
            return [value - 1 for value in tail], True, "bracket_tail"

    digits = [int(value) for value in re.findall(rf"\b[1-{n}]\b", text)]
    if len(digits) >= n:
        tail = digits[-n:]
        if sorted(tail) == list(range(1, n + 1)):
            return [value - 1 for value in tail], True, "digit_tail"

    return list(range(n)), False, f"invalid: {text[-150:]}"

def metrics_from_ranks(gold_ranks):
    ranks = [int(rank) for rank in gold_ranks]
    n = len(ranks)
    if n == 0:
        return {
            "n": 0,
            "hits_at_1": None,
            "recall_at_3": None,
            "mrr": None,
            "ndcg_at_5": None,
        }
    return {
        "n": n,
        "hits_at_1": round(sum(rank == 1 for rank in ranks) / n, 4),
        "recall_at_3": round(sum(rank <= 3 for rank in ranks) / n, 4),
        "mrr": round(sum(1.0 / rank for rank in ranks) / n, 4),
        "ndcg_at_5": round(
            sum(
                1.0 / math.log2(rank + 1)
                for rank in ranks
                if rank <= 5
            ) / n,
            4,
        ),
    }

print("Shared data and metric helpers loaded.")


In [ ]:
def completed_indices(path):
    path = Path(path)
    if not path.exists():
        return set()
    return {
        int(row["instance_index"])
        for row in load_jsonl(path)
        if "instance_index" in row
    }

def save_summary_csv(path, summary):
    pd.DataFrame([summary]).to_csv(path, index=False, encoding="utf-8")

def release_model():
    for name in ["model", "tokenizer", "trainer"]:
        if name in globals():
            try:
                del globals()[name]
            except Exception:
                pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Checkpoint helpers loaded.")


# 3. Dataset sanity checks

These checks are placed before model evaluation because they verify that the benchmark itself does not create an artificial positional advantage.


In [ ]:
def dataset_manifest():
    rows = []
    for split_name, mapping in [("train", TRAIN_FILES), ("test", TEST_FILES)]:
        for strategy, filename in mapping.items():
            path = DATA_DIR / filename
            rows.append({
                "split": split_name,
                "strategy": strategy,
                "file": filename,
                "exists": path.exists(),
                "instances": len(load_jsonl(path)) if path.exists() else None,
            })
    return pd.DataFrame(rows)

dataset_manifest()


In [ ]:
def gold_position_distribution(path):
    counts = Counter()
    instances = load_jsonl(path)
    for instance in instances:
        counts[get_gold_idx(instance) + 1] += 1
    return {
        "file": Path(path).name,
        "total": len(instances),
        "positions_1_to_5": {
            position: counts[position] for position in range(1, 6)
        },
    }

for strategy, filename in TEST_FILES.items():
    print(gold_position_distribution(DATA_DIR / filename))


### Recorded sanity-check result

For the full `domain_lexical` test file, the gold response appeared **55–79 times in each slot** across 330 instances. This was sufficiently balanced to rule out gold-position imbalance as the explanation for the later position-bias finding. The exact counts are regenerated by the cell above.


# 4. Original single-pass LLM baselines

These experiments reproduce the stage before permutation-based bias control.

## Runtime boundary A

Load and evaluate one model per fresh Colab session. Save its outputs, release the model, and restart before loading another model.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import login
from google.colab import userdata

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)

def load_quantized_model(model_name, trust_remote_code=False):
    global model, tokenizer
    release_model()

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        token=hf_token,
        trust_remote_code=trust_remote_code,
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        token=hf_token,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=trust_remote_code,
    )
    model.eval()
    print("Loaded:", model_name)
    print("Device:", next(model.parameters()).device)


In [ ]:
SINGLE_PASS_SYSTEM = (
    "You are a strict Persian social-commerce response-ranking evaluator. "
    "Rank seller replies by relevance and usefulness for the customer's message. "
    "Prefer replies that directly answer the customer's intent, match the product/domain, "
    "and are specific rather than generic. "
    "Return only a JSON array of candidate numbers ordered from best to worst. "
    "Use each number exactly once. No explanation."
)

def build_single_pass_messages(query, candidate_texts, family):
    candidate_block = "\n".join(
        f"{index + 1}. {text}"
        for index, text in enumerate(candidate_texts)
    )
    user_content = (
        f"Customer message:\n{query}\n\n"
        f"Candidate seller replies:\n{candidate_block}\n\n"
        f"Return only JSON, for example: "
        f"{[index + 1 for index in range(len(candidate_texts))]}"
    )

    if family == "gemma":
        # Corrected Gemma run: no separate system role.
        return [{
            "role": "user",
            "content": SINGLE_PASS_SYSTEM + "\n\n" + user_content,
        }]

    return [
        {"role": "system", "content": SINGLE_PASS_SYSTEM},
        {"role": "user", "content": user_content},
    ]

def generate_single(messages, max_new_tokens=64):
    device = next(model.parameters()).device
    encoded = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    inputs = {
        key: value.to(device) if hasattr(value, "to") else value
        for key, value in dict(encoded).items()
    }
    prompt_length = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    raw = tokenizer.decode(
        outputs[0][prompt_length:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    ).strip()
    return raw

def rank_single_pass(instance, family):
    query = extract_query(instance)
    candidates = extract_candidates(instance)
    texts = [candidate_text(candidate) for candidate in candidates]

    raw = generate_single(
        build_single_pass_messages(query, texts, family)
    )
    ranking, valid, status = parse_ranking_bracket(raw, len(texts))
    gold_idx = get_gold_idx(instance)
    rank_by_candidate = {
        candidate_idx: rank + 1
        for rank, candidate_idx in enumerate(ranking)
    }

    return {
        "query": query,
        "candidate_texts": texts,
        "gold_idx": gold_idx,
        "gold_rank": rank_by_candidate[gold_idx],
        "parsed_ranking": [value + 1 for value in ranking],
        "parse_valid": valid,
        "parse_status": status,
        "raw_output": raw,
    }


In [ ]:
def evaluate_single_pass_checkpointed(
    model_tag,
    family,
    strategy,
    max_instances,
    checkpoint_every=10,
):
    dataset = load_jsonl(DATA_DIR / TEST_FILES[strategy])[:max_instances]
    detail_path = RESULTS_DIR / f"{model_tag}_{strategy}_details.jsonl"
    summary_path = RESULTS_DIR / f"{model_tag}_{strategy}_summary.csv"
    done = completed_indices(detail_path)

    print(
        f"{model_tag} | {strategy} | target={len(dataset)} | "
        f"already completed={len(done)}"
    )

    for index, instance in enumerate(dataset):
        if index in done:
            continue

        start = time.time()
        try:
            result = rank_single_pass(instance, family)
            row = {
                "model_tag": model_tag,
                "strategy": strategy,
                "instance_index": index,
                **result,
                "elapsed_sec": round(time.time() - start, 3),
            }
        except Exception as error:
            candidate_count = len(extract_candidates(instance))
            row = {
                "model_tag": model_tag,
                "strategy": strategy,
                "instance_index": index,
                "query": extract_query(instance),
                "candidate_texts": [
                    candidate_text(candidate)
                    for candidate in extract_candidates(instance)
                ],
                "gold_idx": get_gold_idx(instance),
                "gold_rank": candidate_count,
                "parsed_ranking": None,
                "parse_valid": False,
                "parse_status": "exception",
                "raw_output": "",
                "error": repr(error),
                "elapsed_sec": round(time.time() - start, 3),
            }

        append_jsonl(detail_path, row)

        if (index + 1) % checkpoint_every == 0 or index + 1 == len(dataset):
            rows = load_jsonl(detail_path)
            summary = {
                "model_tag": model_tag,
                "family": family,
                "strategy": strategy,
                **metrics_from_ranks([row["gold_rank"] for row in rows]),
                "invalid_outputs": sum(
                    not bool(row.get("parse_valid")) for row in rows
                ),
                "details_file": str(detail_path),
            }
            save_summary_csv(summary_path, summary)
            print(index + 1, summary)

    return pd.read_csv(summary_path).iloc[0].to_dict()


## 4.1 Llama-3.1-8B-Instruct — subset-50 listwise evaluation

The original file evaluated 50 instances for each strategy and saved detailed, error, checkpoint, and summary files.


In [ ]:
LLAMA_MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
LLAMA_TAG = "llama3_1_8b_instruct_listwise_subset50"

if RUN_HEAVY_EXPERIMENTS:
    load_quantized_model(LLAMA_MODEL_NAME)
    llama_results = [
        evaluate_single_pass_checkpointed(
            model_tag=LLAMA_TAG,
            family="llama",
            strategy=strategy,
            max_instances=50,
        )
        for strategy in ["random", "domain", "domain_lexical"]
    ]
    display(pd.DataFrame(llama_results))


### Llama result note

The Llama code was present, but its numerical summary CSV was not embedded in the supplied source files. The completed results were downloaded locally. Expected summary filenames:

- `llama3_1_8b_instruct_listwise_subset50_random_summary.csv`
- `llama3_1_8b_instruct_listwise_subset50_domain_summary.csv`
- `llama3_1_8b_instruct_listwise_subset50_domain_lexical_summary.csv`


## 4.2 Gemma-2-9B-IT — full-330 single-pass evaluation

The corrected Gemma run used a user-only chat structure because the Gemma chat template did not accept a separate system role.


In [ ]:
GEMMA_MODEL_NAME = "google/gemma-2-9b-it"
GEMMA_SINGLE_TAG = "gemma2_9b_it_listwise_full330_single_pass"

if RUN_HEAVY_EXPERIMENTS:
    load_quantized_model(GEMMA_MODEL_NAME)
    gemma_single_results = [
        evaluate_single_pass_checkpointed(
            model_tag=GEMMA_SINGLE_TAG,
            family="gemma",
            strategy=strategy,
            max_instances=330,
        )
        for strategy in ["random", "domain", "domain_lexical"]
    ]
    display(pd.DataFrame(gemma_single_results))


### Recorded Gemma single-pass result

| Model | Protocol | Strategy | n | Hits@1 |
|---|---|---:|---:|---:|
| Gemma-2-9B-IT | single-pass listwise | domain_lexical | 330 | **0.491** |

This remains the historical original-order result, not the position-bias-controlled score.


# 5. Initial heavy systematic prompt and collapse diagnostic

The first systematic prompt used five worked examples plus a Persian grocery vocabulary block. Although outputs were usually parseable, the model frequently reproduced a fixed demonstration-like order instead of ranking by content.

The long prompt itself remains in `dorna2_pershop.py`; the essential diagnostic is preserved here to avoid duplicating a large obsolete prompt in the main notebook.


In [ ]:
def ranking_frequency_diagnostic(
    details_csv,
    ranking_column="parsed_ranking",
):
    dataframe = pd.read_csv(details_csv)
    counts = Counter(dataframe[ranking_column].astype(str))
    ranking, count = counts.most_common(1)[0]
    return {
        "file": str(details_csv),
        "most_common_ranking": ranking,
        "count": count,
        "n": len(dataframe),
        "rate": round(count / len(dataframe), 4),
    }

# Example:
# for strategy in ["random", "domain", "domain_lexical"]:
#     path = RESULTS_DIR / (
#         "gemma2_9b_it_rankgpt_format_all_strategies_subset50_colab_"
#         f"{strategy}_details.csv"
#     )
#     print(ranking_frequency_diagnostic(path))


### Recorded collapse result

The fixed ranking `[4] > [2] > [3] > [1] > [5]` appeared in approximately **60–78%** of outputs, depending on strategy. This diagnosed prompt anchoring and motivated the lean two-example prompt.


# 6. Position-bias diagnostic: zero-shot PSC

Permutation Self-Consistency (PSC) shuffles the same candidates, ranks each permutation, maps predictions back to original candidate identities, and aggregates them using Borda count.

The original zero-shot diagnostic used:

- Gemma-2-9B-IT;
- `domain_lexical`;
- n=50;
- **k=3** shuffles;
- no few-shot examples;
- winner consistency as a stability measure.


In [ ]:
def build_zero_shot_messages(query, shuffled_candidates):
    candidate_block = "\n".join(
        f"[{index + 1}] {candidate_text(candidate)}"
        for index, candidate in enumerate(shuffled_candidates)
    )
    content = (
        "You are RankGPT, an intelligent assistant that ranks seller responses "
        "based on their relevance to a Persian customer message.\n\n"
        f"Customer message: {query}\n\n"
        f"Candidate seller responses:\n{candidate_block}\n\n"
        "Rank the 5 candidate seller responses from best to worst.\n"
        "All five identifiers must appear exactly once.\n"
        "Use this exact output format:\n"
        "[4] > [2] > [3] > [1] > [5]\n\n"
        "Only respond with the ranking results. Do not explain."
    )
    return [{"role": "user", "content": content}]

def generate_batch(
    messages_list,
    max_new_tokens=64,
    max_length=1536,
):
    device = next(model.parameters()).device
    prompts = [
        tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
        )
        for messages in messages_list
    ]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length,
    ).to(device)

    input_width = inputs["input_ids"].shape[1]
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = [
        tokenizer.decode(
            outputs[index][input_width:],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        ).strip()
        for index in range(len(messages_list))
    ]

    del inputs, outputs
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return decoded


In [ ]:
def rank_instance_psc(
    instance,
    build_messages_fn,
    generate_fn,
    k,
    retry_messages_fn=None,
):
    query = extract_query(instance)
    candidates = extract_candidates(instance)
    candidate_count = len(candidates)
    gold_idx = get_gold_idx(instance)

    permutations = []
    shuffled_batches = []
    message_batches = []

    for _ in range(k):
        permutation = list(range(candidate_count))
        random.shuffle(permutation)
        shuffled = [candidates[index] for index in permutation]

        permutations.append(permutation)
        shuffled_batches.append(shuffled)
        message_batches.append(build_messages_fn(query, shuffled))

    raw_outputs = generate_fn(message_batches)

    if retry_messages_fn is not None:
        retry_positions = []
        retry_messages = []

        for position, raw in enumerate(raw_outputs):
            _, valid, _ = parse_ranking_bracket(raw, candidate_count)
            if not valid:
                retry_positions.append(position)
                retry_messages.append(
                    retry_messages_fn(
                        query,
                        shuffled_batches[position],
                        raw,
                    )
                )

        if retry_messages:
            retry_outputs = generate_fn(retry_messages)
            for position, retry_raw in zip(
                retry_positions,
                retry_outputs,
            ):
                raw_outputs[position] = (
                    f"ATTEMPT_1: {raw_outputs[position]}\n"
                    f"ATTEMPT_2: {retry_raw}"
                )

    borda_scores = [0] * candidate_count
    per_shuffle_top = []
    per_shuffle_rankings = []
    per_shuffle_gold_ranks = []
    per_shuffle_valid = []
    parse_statuses = []

    for permutation, raw in zip(permutations, raw_outputs):
        ranking_slots, valid, status = parse_ranking_bracket(
            raw,
            candidate_count,
        )
        per_shuffle_valid.append(valid)
        parse_statuses.append(status)

        if valid:
            ranking_original = [
                permutation[slot] for slot in ranking_slots
            ]
            for rank_position, original_idx in enumerate(
                ranking_original
            ):
                borda_scores[original_idx] += rank_position

            per_shuffle_top.append(ranking_original[0])
            per_shuffle_gold_ranks.append(
                ranking_original.index(gold_idx) + 1
            )
        else:
            ranking_original = list(range(candidate_count))
            for original_idx in range(candidate_count):
                borda_scores[original_idx] += candidate_count
            per_shuffle_gold_ranks.append(candidate_count)

        per_shuffle_rankings.append(
            [index + 1 for index in ranking_original]
        )

    aggregated = sorted(
        range(candidate_count),
        key=lambda index: borda_scores[index],
    )

    return {
        "query": query,
        "candidate_texts": [
            candidate_text(candidate) for candidate in candidates
        ],
        "gold_idx": gold_idx,
        "gold_rank": aggregated.index(gold_idx) + 1,
        "aggregated_ranking": [
            index + 1 for index in aggregated
        ],
        "borda_scores": borda_scores,
        "k_shuffles": k,
        "valid_shuffles": sum(per_shuffle_valid),
        "winner_consistent": (
            len(set(per_shuffle_top)) == 1
            if per_shuffle_top else False
        ),
        "permutations": permutations,
        "per_shuffle_rankings": per_shuffle_rankings,
        "per_shuffle_gold_ranks": per_shuffle_gold_ranks,
        "per_shuffle_valid": per_shuffle_valid,
        "parse_statuses": parse_statuses,
        "raw_outputs": raw_outputs,
    }


In [ ]:
def evaluate_psc_checkpointed(
    experiment_tag,
    strategy,
    max_instances,
    k,
    build_messages_fn,
    generate_fn,
    retry_messages_fn=None,
    checkpoint_every=10,
):
    instances = load_jsonl(
        DATA_DIR / TEST_FILES[strategy]
    )[:max_instances]

    detail_path = (
        RESULTS_DIR /
        f"{experiment_tag}_{strategy}_details.jsonl"
    )
    summary_path = (
        RESULTS_DIR /
        f"{experiment_tag}_{strategy}_summary.csv"
    )
    done = completed_indices(detail_path)

    print(
        f"{experiment_tag} | {strategy} | "
        f"target={len(instances)} | completed={len(done)}"
    )

    for index, instance in enumerate(instances):
        if index in done:
            continue

        start = time.time()
        try:
            result = rank_instance_psc(
                instance=instance,
                build_messages_fn=build_messages_fn,
                generate_fn=generate_fn,
                k=k,
                retry_messages_fn=retry_messages_fn,
            )
            row = {
                "experiment_tag": experiment_tag,
                "strategy": strategy,
                "instance_index": index,
                **result,
                "elapsed_sec": round(time.time() - start, 3),
            }
        except Exception as error:
            candidate_count = len(extract_candidates(instance))
            row = {
                "experiment_tag": experiment_tag,
                "strategy": strategy,
                "instance_index": index,
                "query": extract_query(instance),
                "candidate_texts": [
                    candidate_text(candidate)
                    for candidate in extract_candidates(instance)
                ],
                "gold_idx": get_gold_idx(instance),
                "gold_rank": candidate_count,
                "aggregated_ranking": list(
                    range(1, candidate_count + 1)
                ),
                "borda_scores": None,
                "k_shuffles": k,
                "valid_shuffles": 0,
                "winner_consistent": False,
                "permutations": None,
                "per_shuffle_rankings": None,
                "per_shuffle_gold_ranks": [candidate_count] * k,
                "per_shuffle_valid": [False] * k,
                "parse_statuses": ["exception"] * k,
                "raw_outputs": None,
                "error": repr(error),
                "elapsed_sec": round(time.time() - start, 3),
            }

        append_jsonl(detail_path, row)

        if (
            (index + 1) % checkpoint_every == 0
            or index + 1 == len(instances)
        ):
            rows = load_jsonl(detail_path)
            gold_ranks = [row["gold_rank"] for row in rows]
            all_shuffle_ranks = [
                rank
                for row in rows
                for rank in row.get(
                    "per_shuffle_gold_ranks",
                    [],
                )
            ]

            summary = {
                "experiment_tag": experiment_tag,
                "strategy": strategy,
                "max_instances": max_instances,
                "k_shuffles": k,
                **metrics_from_ranks(gold_ranks),
                "winner_consistency": round(
                    sum(
                        bool(row.get("winner_consistent"))
                        for row in rows
                    ) / len(rows),
                    4,
                ),
                "valid_shuffle_rate": round(
                    sum(
                        int(row.get("valid_shuffles", 0))
                        for row in rows
                    ) / max(1, len(rows) * k),
                    4,
                ),
                "per_shuffle_hits_at_1": round(
                    sum(rank == 1 for rank in all_shuffle_ranks)
                    / max(1, len(all_shuffle_ranks)),
                    4,
                ),
                "details_file": str(detail_path),
            }

            save_summary_csv(summary_path, summary)
            print(index + 1, summary)

    return pd.read_csv(summary_path).iloc[0].to_dict()


In [ ]:
ZERO_SHOT_PSC_TAG = (
    "gemma2_9b_it_zero_shot_psc_k3_domain_lexical"
)

if RUN_HEAVY_EXPERIMENTS:
    load_quantized_model(GEMMA_MODEL_NAME)
    zero_shot_psc_result = evaluate_psc_checkpointed(
        experiment_tag=ZERO_SHOT_PSC_TAG,
        strategy="domain_lexical",
        max_instances=50,
        k=3,
        build_messages_fn=build_zero_shot_messages,
        generate_fn=generate_batch,
    )
    zero_shot_psc_result


### Recorded position-bias result

| Quantity | Value |
|---|---:|
| Original-order single-pass Hits@1, n=330 | **0.491** |
| Shuffled non-aggregated Hits@1, 50 × 3 generations | **0.3667** |
| Winner consistency across the three shuffles | **18%** |

The source files did not embed the aggregated Borda Hits@1 for this diagnostic. Therefore, the notebook reports the available per-shuffle value precisely rather than mislabelling it as aggregated performance.


# 7. Lean systematic prompt + PSC

The final lean prompt uses two demonstrations, one direct-answer rule, and no domain glossary.

## Actual original protocol

- n=50 per strategy;
- **k=10** for the final lean experiment;
- retry on invalid output in the final domain-lexical implementation;
- Borda aggregation.

Later CoP, RAG-ICL, Merge, and fair-condition Dorna experiments used k=5. The notebook preserves those separate settings.


In [ ]:
LEAN_FEW_SHOT_TURNS = [
    (
        "برند کاله دارید؟\n"
        "[1] چه برندی مد نظرتونه؟ دوغ گازدار ۱۵۰۰ میلی‌لیتری کاله\n"
        "[2] دوغ گازدار ۱۵۰۰ میلی‌لیتری کاله\n"
        "[3] گازدار باشه یا بدون گاز؟ دوغ گازدار ۱۵۰۰ میلی‌لیتری کاله\n"
        "[4] دوغ با طعم نعناع بدون گاز ۱۵۰۰ میلی‌لیتری عالیس\n"
        "[5] چه برندی مد نظرتونه؟ برندهای زیر موجودند: دوغ گازدار ۱۵۰۰ میلی‌لیتری کاله",
        "[2] > [3] > [1] > [5] > [4]",
    ),
    (
        "ماست بدون چربی دارید؟\n"
        "[1] ماست پرچرب موسیر پاک ۲ کیلوگرمی\n"
        "[2] ماست صفر درصد چربی لاکتیویا ۹۰۰ گرمی کاله\n"
        "[3] چه برندی مد نظرتونه؟ ماست کم‌چرب ۱.۵٪ دامداران ۹۰۰ گرمی\n"
        "[4] ماست چکیده موسیر ۵۰۰ گرمی چوپان\n"
        "[5] شیر کم‌چرب ۲۰۰ میلی‌لیتری فرادما میهن",
        "[2] > [3] > [4] > [1] > [5]",
    ),
]

LEAN_INSTRUCTION = (
    "You are RankGPT, ranking Persian grocery-seller responses "
    "by relevance to a customer message. Rule: a response that "
    "directly gives the exact requested product outranks one that "
    "asks a clarifying question first, even if that question names "
    "the same product. Output format: [] > [] > [] > [] > [], "
    "using all 5 identifiers exactly once. No explanation."
)

def build_lean_messages(query, shuffled_candidates):
    candidate_block = "\n".join(
        f"[{index + 1}] {candidate_text(candidate)}"
        for index, candidate in enumerate(shuffled_candidates)
    )

    messages = [
        {"role": "user", "content": LEAN_INSTRUCTION},
        {"role": "model", "content": "Understood. Ready to rank."},
    ]

    for example, ranking in LEAN_FEW_SHOT_TURNS:
        messages.append({"role": "user", "content": example})
        messages.append({"role": "model", "content": ranking})

    messages.append({
        "role": "user",
        "content": (
            "This is a new, independent instance. "
            "Do not copy the example rankings.\n\n"
            f"{query}\n{candidate_block}"
        ),
    })
    return messages

def build_lean_retry_messages(
    query,
    shuffled_candidates,
    previous_raw,
):
    candidate_block = "\n".join(
        f"[{index + 1}] {candidate_text(candidate)}"
        for index, candidate in enumerate(shuffled_candidates)
    )

    return [{
        "role": "user",
        "content": (
            f"Your previous answer was invalid:\n{previous_raw}\n\n"
            "Correct it. Return only a complete ranking in "
            "[] > [] format. Use [1], [2], [3], [4], and [5] "
            "exactly once.\n\n"
            f"Customer message:\n{query}\n\n"
            f"Candidate seller responses:\n{candidate_block}"
        ),
    }]


In [ ]:
LEAN_TAG = "gemma2_9b_it_lean_prompt_psc_k10"

if RUN_HEAVY_EXPERIMENTS:
    load_quantized_model(GEMMA_MODEL_NAME)
    lean_results = [
        evaluate_psc_checkpointed(
            experiment_tag=LEAN_TAG,
            strategy=strategy,
            max_instances=50,
            k=10,
            build_messages_fn=build_lean_messages,
            generate_fn=generate_batch,
            retry_messages_fn=(
                build_lean_retry_messages
                if strategy == "domain_lexical"
                else None
            ),
        )
        for strategy in [
            "random",
            "domain",
            "domain_lexical",
        ]
    ]
    display(pd.DataFrame(lean_results))


### Recorded lean-prompt results

| Strategy | n | k | Hits@1 | Recall@3 | MRR | nDCG@5 | Winner consistency |
|---|---:|---:|---:|---:|---:|---:|---:|
| random | 50 | 10 | **0.96** | 1.00 | 0.977 | 0.983 | 84% |
| domain | 50 | 10 | **0.88** | 1.00 | 0.937 | 0.953 | 72% |
| domain_lexical | 50 | 10 | **0.52** | 0.80 | 0.680 | 0.759 | 30% |

The recorded domain-lexical per-shuffle Hits@1 was 0.50; Borda aggregation raised it to 0.52.


# 8. Chain-of-Prompting + PSC

Protocol: frozen Gemma, n=50 per strategy, k=5, English Plan-and-Solve reasoning over Persian content, and up to 200 generated tokens to avoid truncated reasoning.


In [ ]:
TASK_DECLARATION = (
    "You are ranking Persian seller responses by relevance to a "
    "Persian customer message. Understand the Persian content, "
    "then reason about the ranking in English. Refer to candidates "
    "by number rather than translating the text."
)

PLAN_AND_SOLVE_CHECKLIST = (
    "Checklist:\n"
    "1. Does the candidate match the exact product, brand, or attribute?\n"
    "2. A direct exact answer outranks a clarification-first answer.\n"
    "3. Among direct answers, prefer the response matching more requested attributes.\n"
    "4. Rank all five candidates from best to worst."
)

COP_OUTPUT_RULE = (
    "After brief English reasoning, end with exactly one ranking "
    "line in [n] > [n] > [n] > [n] > [n] format. "
    "Use every identifier once."
)

COP_SYSTEM_CONTENT = (
    f"{TASK_DECLARATION}\n\n"
    f"{PLAN_AND_SOLVE_CHECKLIST}\n\n"
    f"{COP_OUTPUT_RULE}"
)

COP_EXAMPLE_QUERY = "برند کاله دارید؟"
COP_EXAMPLE_CANDIDATES = (
    "[1] چه برندی مد نظرتونه؟ دوغ گازدار ۱۵۰۰ میلی‌لیتری کاله\n"
    "[2] دوغ گازدار ۱۵۰۰ میلی‌لیتری کاله\n"
    "[3] گازدار باشه یا بدون گاز؟ دوغ گازدار ۱۵۰۰ میلی‌لیتری کاله\n"
    "[4] دوغ با طعم نعناع بدون گاز ۱۵۰۰ میلی‌لیتری عالیس\n"
    "[5] چه برندی مد نظرتونه؟ برندهای زیر موجودند: دوغ گازدار ۱۵۰۰ میلی‌لیتری کاله"
)
COP_EXAMPLE_TARGET = (
    "Reasoning: [2] is the most direct matching answer. "
    "[3] matches but asks a clarification. [1] and [5] "
    "unnecessarily ask about the brand. [4] is another brand.\n"
    "Final ranking:\n"
    "[2] > [3] > [1] > [5] > [4]"
)

def build_cop_messages(query, shuffled_candidates):
    candidate_block = "\n".join(
        f"[{index + 1}] {candidate_text(candidate)}"
        for index, candidate in enumerate(shuffled_candidates)
    )

    return [
        {"role": "user", "content": COP_SYSTEM_CONTENT},
        {
            "role": "model",
            "content": (
                "Understood. I will apply the checklist and "
                "end with one ranking line."
            ),
        },
        {
            "role": "user",
            "content": (
                f"Customer message: {COP_EXAMPLE_QUERY}\n\n"
                f"Candidates:\n{COP_EXAMPLE_CANDIDATES}"
            ),
        },
        {"role": "model", "content": COP_EXAMPLE_TARGET},
        {
            "role": "user",
            "content": (
                "This is a new independent instance.\n\n"
                f"Customer message: {query}\n\n"
                f"Candidates:\n{candidate_block}"
            ),
        },
    ]

def generate_cop_batch(messages_list):
    return generate_batch(
        messages_list,
        max_new_tokens=200,
        max_length=1536,
    )


In [ ]:
COP_TAG = "gemma2_9b_it_cop_psc_k5"

if RUN_HEAVY_EXPERIMENTS:
    load_quantized_model(GEMMA_MODEL_NAME)
    cop_results = [
        evaluate_psc_checkpointed(
            experiment_tag=COP_TAG,
            strategy=strategy,
            max_instances=50,
            k=5,
            build_messages_fn=build_cop_messages,
            generate_fn=generate_cop_batch,
        )
        for strategy in [
            "random",
            "domain",
            "domain_lexical",
        ]
    ]
    display(pd.DataFrame(cop_results))


### Recorded CoP result

| Strategy | Hits@1 | Comparison with lean |
|---|---:|---|
| random | **0.96** | tied |
| domain | **0.86** | −0.02 |
| domain_lexical | approximately **0.50** | no gain |

The method required approximately four to five times more inference and is retained as a negative result.


# 9. Self-distilled curriculum LoRA SFT

## Runtime boundary B

Use a fresh Gemma runtime for distillation and training. The SFT adapter changes model state; reload the base model before frozen-model RAG-ICL.

Training order is intentional: `random` → `domain` → `domain_lexical`.


In [ ]:
DISTILL_SUBSET_SIZES = {
    "random": 150,
    "domain": 150,
    "domain_lexical": 250,
}

def distill_strategy(strategy, batch_size=4):
    checkpoint_path = (
        CHECKPOINT_DIR /
        f"distill_{strategy}.jsonl"
    )
    existing = (
        load_jsonl(checkpoint_path)
        if checkpoint_path.exists()
        else []
    )
    completed = {
        int(row["source_index"]) for row in existing
    }
    instances = load_jsonl(
        DATA_DIR / TRAIN_FILES[strategy]
    )[:DISTILL_SUBSET_SIZES[strategy]]
    kept = list(existing)

    pending = [
        (index, instance)
        for index, instance in enumerate(instances)
        if index not in completed
    ]

    for start in range(0, len(pending), batch_size):
        batch = pending[start:start + batch_size]
        metadata = []
        messages = []

        for source_index, instance in batch:
            candidates = extract_candidates(instance)
            permutation = list(range(len(candidates)))
            random.shuffle(permutation)
            shuffled = [
                candidates[index] for index in permutation
            ]
            gold_shuffled_position = permutation.index(
                get_gold_idx(instance)
            )

            metadata.append((
                source_index,
                instance,
                shuffled,
                gold_shuffled_position,
            ))
            messages.append(
                build_cop_messages(
                    extract_query(instance),
                    shuffled,
                )
            )

        raw_outputs = generate_cop_batch(messages)

        for (
            source_index,
            instance,
            shuffled,
            gold_position,
        ), raw in zip(metadata, raw_outputs):
            ranking, valid, _ = parse_ranking_bracket(
                raw,
                len(shuffled),
            )
            if valid and ranking[0] == gold_position:
                kept.append({
                    "strategy": strategy,
                    "source_index": source_index,
                    "query": extract_query(instance),
                    "candidates_shuffled": [
                        candidate_text(candidate)
                        for candidate in shuffled
                    ],
                    "target_text": raw,
                })

        with checkpoint_path.open(
            "w",
            encoding="utf-8",
        ) as output:
            for row in kept:
                output.write(
                    json.dumps(
                        row,
                        ensure_ascii=False,
                    ) + "\n"
                )

        print(
            strategy,
            f"processed={min(start + len(batch), len(pending))}/{len(pending)}",
            f"kept={len(kept)}",
        )

    return kept

if RUN_HEAVY_EXPERIMENTS:
    load_quantized_model(GEMMA_MODEL_NAME)
    kept_random = distill_strategy("random")
    kept_domain = distill_strategy("domain")
    kept_domain_lexical = distill_strategy(
        "domain_lexical"
    )


### Recorded distillation counts

- random: **106 / 150** retained;
- domain: **93 / 150** retained;
- domain_lexical: approximately **87–99 / 250**, depending on shuffle/run.

A format bug was corrected during distillation. Removing a literal example ranking raised the random-strategy keep rate from about 33% to 71%.


In [ ]:
def assemble_curriculum():
    random_rows = load_jsonl(
        CHECKPOINT_DIR / "distill_random.jsonl"
    )
    domain_rows = load_jsonl(
        CHECKPOINT_DIR / "distill_domain.jsonl"
    )
    dlex_rows = load_jsonl(
        CHECKPOINT_DIR /
        "distill_domain_lexical.jsonl"
    )

    curriculum = (
        random_rows +
        domain_rows +
        dlex_rows
    )
    output_path = (
        RESULTS_DIR /
        "sft_curriculum_dataset.jsonl"
    )

    with output_path.open(
        "w",
        encoding="utf-8",
    ) as output:
        for row in curriculum:
            output.write(
                json.dumps(
                    row,
                    ensure_ascii=False,
                ) + "\n"
            )

    print({
        "total": len(curriculum),
        "random": len(random_rows),
        "domain": len(domain_rows),
        "domain_lexical": len(dlex_rows),
    })
    return curriculum

# curriculum_rows = assemble_curriculum()


In [ ]:
from datasets import Dataset
from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from trl import SFTConfig, SFTTrainer

SFT_ADAPTER_PATH = (
    ADAPTER_DIR /
    "gemma_pershop_curriculum_sft"
)

def sft_row_to_text(row):
    candidate_block = "\n".join(
        f"[{index + 1}] {text}"
        for index, text in enumerate(
            row["candidates_shuffled"]
        )
    )

    messages = [
        {"role": "user", "content": COP_SYSTEM_CONTENT},
        {"role": "model", "content": "Understood."},
        {
            "role": "user",
            "content": (
                f"Customer message: {row['query']}\n\n"
                f"Candidates:\n{candidate_block}"
            ),
        },
        {"role": "model", "content": row["target_text"]},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
    )

def make_sft_config(stage_name, epochs):
    return SFTConfig(
        output_dir=str(
            CHECKPOINT_DIR / f"sft_{stage_name}"
        ),
        num_train_epochs=epochs,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=1e-4,
        logging_steps=10,
        save_strategy="epoch",
        fp16=False,
        report_to="none",
        max_length=1024,
        dataset_text_field="text",
    )

if RUN_HEAVY_EXPERIMENTS:
    load_quantized_model(GEMMA_MODEL_NAME)
    model = prepare_model_for_kbit_training(
        model
    )
    model = get_peft_model(
        model,
        LoraConfig(
            r=16,
            lora_alpha=32,
            lora_dropout=0.05,
            target_modules=[
                "q_proj",
                "k_proj",
                "v_proj",
                "o_proj",
            ],
            task_type="CAUSAL_LM",
        ),
    )

    for parameter in model.parameters():
        if parameter.requires_grad:
            parameter.data = (
                parameter.data.to(torch.float32)
            )

    rows = load_jsonl(
        RESULTS_DIR /
        "sft_curriculum_dataset.jsonl"
    )
    stage_rows = {
        name: [
            row for row in rows
            if row["strategy"] == name
        ]
        for name in [
            "random",
            "domain",
            "domain_lexical",
        ]
    }
    stage_datasets = {
        name: Dataset.from_dict({
            "text": [
                sft_row_to_text(row)
                for row in stage_values
            ]
        })
        for name, stage_values
        in stage_rows.items()
    }

    for stage_name, epochs in [
        ("random", 1),
        ("domain", 1),
        ("domain_lexical", 2),
    ]:
        trainer = SFTTrainer(
            model=model,
            args=make_sft_config(
                stage_name,
                epochs,
            ),
            train_dataset=stage_datasets[
                stage_name
            ],
        )
        trainer.train()

    model.save_pretrained(SFT_ADAPTER_PATH)
    tokenizer.save_pretrained(
        SFT_ADAPTER_PATH
    )
    print("Saved:", SFT_ADAPTER_PATH)


## 9.1 SFT-only evaluation

The adapter is reloaded in a fresh runtime and evaluated with lean listwise PSC to isolate parameter adaptation from retrieval.


In [ ]:
SFT_EVAL_TAG = (
    "gemma2_9b_it_curriculum_sft_psc_k5"
)

if RUN_HEAVY_EXPERIMENTS:
    load_quantized_model(GEMMA_MODEL_NAME)
    model = PeftModel.from_pretrained(
        model,
        SFT_ADAPTER_PATH,
    )
    model.eval()

    sft_result = evaluate_psc_checkpointed(
        experiment_tag=SFT_EVAL_TAG,
        strategy="domain_lexical",
        max_instances=50,
        k=5,
        build_messages_fn=build_lean_messages,
        generate_fn=generate_batch,
    )
    sft_result


### Recorded SFT-only result

| Strategy | n | k | Hits@1 | Output validity |
|---|---:|---:|---:|---:|
| domain_lexical | 50 | 5 | **0.48** | **100%** (250/250 generations) |

SFT improved output-format reliability but did not exceed the frozen lean-prompt result.


# 10. Retrieval-Augmented In-Context Learning

## Runtime boundary C

Reload frozen Gemma. Do not use the SFT-modified model from Section 9.

BGE-m3 retrieves solved same-strategy training instances. `RAG_K = 3` was selected after comparing k=1, 2, and 3 on a 20-instance subset.


In [ ]:
from sentence_transformers import (
    SentenceTransformer,
)

RAG_K = 3
retriever = SentenceTransformer(
    "BAAI/bge-m3"
)

def build_knowledge_base():
    knowledge_base = {}

    for strategy, filename in TRAIN_FILES.items():
        instances = load_jsonl(
            DATA_DIR / filename
        )
        queries = [
            extract_query(instance)
            for instance in instances
        ]
        embeddings = retriever.encode(
            queries,
            batch_size=32,
            normalize_embeddings=True,
            show_progress_bar=True,
        )
        knowledge_base[strategy] = {
            "instances": instances,
            "queries": queries,
            "embeddings": np.asarray(
                embeddings
            ),
        }
        print(strategy, len(instances))

    return knowledge_base

# knowledge_base = build_knowledge_base()


In [ ]:
def gold_first_ranking_line(instance):
    candidates = extract_candidates(instance)
    gold_idx = get_gold_idx(instance)
    order = [
        gold_idx,
        *[
            index
            for index in range(len(candidates))
            if index != gold_idx
        ],
    ]
    return " > ".join(
        f"[{index + 1}]"
        for index in order
    )

def retrieve_examples(
    strategy,
    query,
    k,
    exclude_query=None,
):
    knowledge = knowledge_base[strategy]
    query_embedding = retriever.encode(
        [query],
        normalize_embeddings=True,
    )[0]
    similarities = (
        knowledge["embeddings"]
        @ query_embedding
    )
    order = np.argsort(-similarities)

    examples = []
    for index in order:
        if (
            exclude_query is not None
            and knowledge["queries"][index]
            == exclude_query
        ):
            continue

        examples.append(
            knowledge["instances"][index]
        )
        if len(examples) == k:
            break

    return examples

def build_rag_messages(
    strategy,
    query,
    shuffled_candidates,
    k=RAG_K,
):
    messages = [
        {"role": "user", "content": LEAN_INSTRUCTION},
        {
            "role": "model",
            "content": "Understood. Ready to rank.",
        },
    ]

    for example in retrieve_examples(
        strategy,
        query,
        k,
        exclude_query=query,
    ):
        example_block = "\n".join(
            f"[{index + 1}] "
            f"{candidate_text(candidate)}"
            for index, candidate in enumerate(
                extract_candidates(example)
            )
        )
        messages.append({
            "role": "user",
            "content": (
                f"{extract_query(example)}\n"
                f"{example_block}"
            ),
        })
        messages.append({
            "role": "model",
            "content": (
                gold_first_ranking_line(
                    example
                )
            ),
        })

    candidate_block = "\n".join(
        f"[{index + 1}] "
        f"{candidate_text(candidate)}"
        for index, candidate in enumerate(
            shuffled_candidates
        )
    )
    messages.append({
        "role": "user",
        "content": (
            "This is a new, independent instance.\n\n"
            f"{query}\n{candidate_block}"
        ),
    })
    return messages


In [ ]:
RAG_ICL_TAG = (
    "gemma2_9b_it_rag_icl_k3_psc_k5"
)

if RUN_HEAVY_EXPERIMENTS:
    load_quantized_model(GEMMA_MODEL_NAME)
    knowledge_base = build_knowledge_base()

    rag_results = []
    for strategy in [
        "random",
        "domain",
        "domain_lexical",
    ]:
        build_fn = (
            lambda query, candidates, strategy=strategy:
            build_rag_messages(
                strategy,
                query,
                candidates,
                k=RAG_K,
            )
        )
        rag_results.append(
            evaluate_psc_checkpointed(
                experiment_tag=RAG_ICL_TAG,
                strategy=strategy,
                max_instances=50,
                k=5,
                build_messages_fn=build_fn,
                generate_fn=generate_batch,
            )
        )

    display(pd.DataFrame(rag_results))


### Recorded RAG-ICL results

| Strategy | Hits@1 | MRR | Winner consistency |
|---|---:|---:|---:|
| random | **0.94** | 0.970 | **100%** |
| domain | **0.88** | 0.937 | **84%** |
| domain_lexical | **0.52** | 0.680 | **42%** |

RAG-ICL improved stability but not top-1 accuracy.


# 11. Merge: RAG-augmented SFT

The Merge method trains Gemma on examples that already contain retrieved same-strategy neighbours. Retrieval is also present at inference.

## Runtime boundary D

Use a fresh base Gemma and a fresh LoRA adapter. Do not continue from the Section 9 adapter.


In [ ]:
MERGE_RAG_K = 2
MERGE_ADAPTER_PATH = (
    ADAPTER_DIR /
    "gemma_pershop_rag_sft"
)

def build_merge_training_text(
    strategy,
    instance,
    self_index,
):
    query = extract_query(instance)
    knowledge = knowledge_base[strategy]
    query_embedding = retriever.encode(
        [query],
        normalize_embeddings=True,
    )[0]
    similarities = (
        knowledge["embeddings"]
        @ query_embedding
    )
    neighbours = [
        index
        for index in np.argsort(-similarities)
        if index != self_index
    ][:MERGE_RAG_K]

    messages = [
        {"role": "user", "content": LEAN_INSTRUCTION},
        {
            "role": "model",
            "content": "Understood. Ready to rank.",
        },
    ]

    for neighbour_index in neighbours:
        example = knowledge["instances"][
            neighbour_index
        ]
        example_block = "\n".join(
            f"[{index + 1}] "
            f"{candidate_text(candidate)}"
            for index, candidate in enumerate(
                extract_candidates(example)
            )
        )
        messages.append({
            "role": "user",
            "content": (
                f"{extract_query(example)}\n"
                f"{example_block}"
            ),
        })
        messages.append({
            "role": "model",
            "content": (
                gold_first_ranking_line(
                    example
                )
            ),
        })

    target_block = "\n".join(
        f"[{index + 1}] "
        f"{candidate_text(candidate)}"
        for index, candidate in enumerate(
            extract_candidates(instance)
        )
    )
    messages.append({
        "role": "user",
        "content": (
            "This is a new, independent instance.\n\n"
            f"{query}\n{target_block}"
        ),
    })
    messages.append({
        "role": "model",
        "content": (
            gold_first_ranking_line(
                instance
            )
        ),
    })

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
    )

def build_merge_datasets():
    datasets_by_strategy = {}

    for strategy in [
        "random",
        "domain",
        "domain_lexical",
    ]:
        limit = (
            250
            if strategy == "domain_lexical"
            else 150
        )
        instances = (
            knowledge_base[strategy][
                "instances"
            ][:limit]
        )
        texts = [
            build_merge_training_text(
                strategy,
                instance,
                index,
            )
            for index, instance
            in enumerate(instances)
        ]
        datasets_by_strategy[strategy] = (
            Dataset.from_dict({
                "text": texts
            })
        )
        print(strategy, len(texts))

    return datasets_by_strategy


In [ ]:
def make_merge_sft_config(
    stage_name,
    epochs,
):
    return SFTConfig(
        output_dir=str(
            CHECKPOINT_DIR /
            f"merge_{stage_name}"
        ),
        num_train_epochs=epochs,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=1e-4,
        logging_steps=10,
        save_strategy="epoch",
        fp16=False,
        report_to="none",
        max_length=1536,
        dataset_text_field="text",
    )

if RUN_HEAVY_EXPERIMENTS:
    load_quantized_model(GEMMA_MODEL_NAME)
    knowledge_base = build_knowledge_base()
    merge_datasets = build_merge_datasets()

    model = prepare_model_for_kbit_training(
        model
    )
    model = get_peft_model(
        model,
        LoraConfig(
            r=16,
            lora_alpha=32,
            lora_dropout=0.05,
            target_modules=[
                "q_proj",
                "k_proj",
                "v_proj",
                "o_proj",
            ],
            task_type="CAUSAL_LM",
        ),
    )

    for parameter in model.parameters():
        if parameter.requires_grad:
            parameter.data = (
                parameter.data.to(
                    torch.float32
                )
            )

    for stage_name, epochs in [
        ("random", 1),
        ("domain", 1),
        ("domain_lexical", 2),
    ]:
        trainer = SFTTrainer(
            model=model,
            args=make_merge_sft_config(
                stage_name,
                epochs,
            ),
            train_dataset=merge_datasets[
                stage_name
            ],
        )
        trainer.train()

    model.save_pretrained(
        MERGE_ADAPTER_PATH
    )
    tokenizer.save_pretrained(
        MERGE_ADAPTER_PATH
    )


In [ ]:
def build_merge_eval_messages(
    strategy,
    query,
    shuffled_candidates,
):
    return build_rag_messages(
        strategy=strategy,
        query=query,
        shuffled_candidates=shuffled_candidates,
        k=MERGE_RAG_K,
    )

MERGE_TAG = (
    "gemma2_9b_it_rag_augmented_sft_psc_k5"
)

if RUN_HEAVY_EXPERIMENTS:
    load_quantized_model(GEMMA_MODEL_NAME)
    model = PeftModel.from_pretrained(
        model,
        MERGE_ADAPTER_PATH,
    )
    model.eval()
    knowledge_base = build_knowledge_base()

    merge_results = []
    for strategy, max_instances in [
        ("random", 50),
        ("domain", 330),
        ("domain_lexical", 330),
    ]:
        build_fn = (
            lambda query, candidates, strategy=strategy:
            build_merge_eval_messages(
                strategy,
                query,
                candidates,
            )
        )
        merge_results.append(
            evaluate_psc_checkpointed(
                experiment_tag=MERGE_TAG,
                strategy=strategy,
                max_instances=max_instances,
                k=5,
                build_messages_fn=build_fn,
                generate_fn=generate_batch,
                checkpoint_every=30,
            )
        )

    display(pd.DataFrame(merge_results))


### Recorded Merge results

| Strategy | n | Hits@1 | MRR | nDCG@5 | Winner consistency |
|---|---:|---:|---:|---:|---:|
| random | 50 | **0.94** | 0.970 | 0.977 | 100% |
| domain | 330 | **0.827** | 0.891 | 0.918 | 74.5% |
| domain_lexical | 330 | **0.594** | 0.730 | 0.797 | 35.5% |

The domain-lexical confirmation path increased from about **0.47 at n=30**, to **0.567 at an intermediate run**, and finally **0.594 at n=330**. The full run confirmed that the gain over the approximately 0.52 ceiling was not a subset-50 fluctuation.


# 12. Dorna2 exploratory protocols

Before the fair-condition comparison, the original Dorna investigation used three protocols:

1. pointwise scoring from 0 to 10;
2. single-pass listwise ranking;
3. Persian bidirectional pairwise PRP.

They are preserved as methodological history but are not directly comparable with Gemma PSC because their ranking structures and bias controls differ.


### Recorded original Dorna2 results

| Protocol | Strategy | Hits@1 | Interpretation |
|---|---|---:|---|
| pointwise, single-pass | domain_lexical | **0.3121** | exploratory |
| listwise, single-pass | domain_lexical | **0.2939** | exploratory |
| bidirectional pairwise PRP | domain_lexical | stored in local CSV | separate pairwise method |

The pairwise experiment also measured whether the same candidate won after swapping the A/B presentation order.


In [ ]:
def extract_score_0_to_10(text):
    cleaned = normalize_digits(text).strip()
    matches = re.findall(
        r"(?<!\d)(?:10(?:\.0+)?|[0-9](?:\.[0-9]+)?)(?!\d)",
        cleaned,
    )
    if not matches:
        return None

    value = float(matches[0])
    return value if 0 <= value <= 10 else None

def build_dorna_pointwise_messages(
    query,
    candidate,
):
    return [
        {
            "role": "system",
            "content": (
                "You are a strict scoring machine. "
                "Return only one number from 0 to 10. "
                "No explanation."
            ),
        },
        {
            "role": "user",
            "content": (
                "Score this seller response for relevance "
                "to the Persian customer message.\n\n"
                f"Customer message:\n{query}\n\n"
                f"Seller response:\n{candidate}\n\n"
                "0 = completely irrelevant; "
                "10 = perfectly relevant."
            ),
        },
    ]


In [ ]:
from itertools import combinations

PERSIAN_SYSTEM_ROLE = (
    "شما یک دستیار فارسی‌زبان مفید هستید."
)

PRP_TEMPLATE = (
    "شما باید مشخص کنید کدام پاسخ فروشنده برای پیام مشتری مناسب‌تر است.\n\n"
    "قانون: پاسخی که مستقیماً محصول درخواستی را معرفی می‌کند، "
    "بر پاسخی که ابتدا سؤال می‌پرسد برتری دارد، حتی اگر آن سؤال "
    "همان محصول را نام ببرد.\n\n"
    "فقط با یک حرف پاسخ دهید: A یا B.\n\n"
    "پیام مشتری: {query}\n"
    "پاسخ A: {response_a}\n"
    "پاسخ B: {response_b}\n"
    "پاسخ بهتر:"
)

def build_prp_messages(
    query,
    response_a,
    response_b,
):
    return [
        {
            "role": "system",
            "content": PERSIAN_SYSTEM_ROLE,
        },
        {
            "role": "user",
            "content": PRP_TEMPLATE.format(
                query=query,
                response_a=response_a,
                response_b=response_b,
            ),
        },
    ]

def parse_ab_verdict(raw):
    match = re.search(
        r"\b(A|B)\b",
        raw.strip().upper(),
    )
    return match.group(1) if match else None

def rank_instance_bidirectional_prp(
    instance,
    generate_verdict_fn,
):
    query = extract_query(instance)
    candidates = extract_candidates(instance)
    candidate_count = len(candidates)
    scores = [0.0] * candidate_count
    consistent_pairs = 0
    invalid_pairs = 0

    for first, second in combinations(
        range(candidate_count),
        2,
    ):
        first_text = candidate_text(
            candidates[first]
        )
        second_text = candidate_text(
            candidates[second]
        )

        forward = parse_ab_verdict(
            generate_verdict_fn(
                build_prp_messages(
                    query,
                    first_text,
                    second_text,
                )
            )
        )
        backward = parse_ab_verdict(
            generate_verdict_fn(
                build_prp_messages(
                    query,
                    second_text,
                    first_text,
                )
            )
        )

        if forward is None or backward is None:
            scores[first] += 0.5
            scores[second] += 0.5
            invalid_pairs += 1
            continue

        forward_winner = (
            first if forward == "A"
            else second
        )
        backward_winner = (
            second if backward == "A"
            else first
        )

        if forward_winner == backward_winner:
            scores[forward_winner] += 1.0
            consistent_pairs += 1
        else:
            scores[first] += 0.5
            scores[second] += 0.5

    ranking = sorted(
        range(candidate_count),
        key=lambda index: -scores[index],
    )

    return {
        "gold_rank": (
            ranking.index(
                get_gold_idx(instance)
            ) + 1
        ),
        "aggregated_ranking": [
            index + 1 for index in ranking
        ],
        "win_scores": scores,
        "bidirectional_consistency": (
            consistent_pairs
            / math.comb(candidate_count, 2)
        ),
        "invalid_pairs": invalid_pairs,
    }


# 13. Dorna2 fair-condition evaluation

## Runtime boundary E

Use a fresh runtime and load only Dorna2.

The fair comparison uses listwise ranking, the lean two-example structure, PSC with k=5, Borda aggregation, n=50 per strategy, Dorna's official Llama-style chat roles, and Persian instructions.


In [ ]:
DORNA_MODEL_NAME = (
    "PartAI/Dorna2-Llama3.1-8B-Instruct"
)
DORNA_FAIR_TAG = (
    "dorna2_fair_lean_psc_k5"
)

DORNA_INSTRUCTION = (
    "شما پاسخ‌های فروشنده را بر اساس ارتباط با پیام مشتری "
    "رتبه‌بندی می‌کنید.\n"
    "قانون: پاسخی که مستقیماً محصول درخواستی را ارائه می‌دهد، "
    "بر پاسخی که ابتدا سؤال روشن‌کننده می‌پرسد برتری دارد، "
    "حتی اگر آن سؤال همان محصول را نام ببرد.\n"
    "قالب خروجی: [] > [] > [] > [] > [] — "
    "هر پنج شناسه دقیقاً یک بار. بدون توضیح."
)

def build_dorna_fair_messages(
    query,
    shuffled_candidates,
):
    candidate_block = "\n".join(
        f"[{index + 1}] "
        f"{candidate_text(candidate)}"
        for index, candidate in enumerate(
            shuffled_candidates
        )
    )

    messages = [
        {
            "role": "system",
            "content": PERSIAN_SYSTEM_ROLE,
        },
        {
            "role": "user",
            "content": DORNA_INSTRUCTION,
        },
        {
            "role": "assistant",
            "content": (
                "متوجه شدم. آماده رتبه‌بندی هستم."
            ),
        },
    ]

    for example, ranking in LEAN_FEW_SHOT_TURNS:
        messages.append({
            "role": "user",
            "content": example,
        })
        messages.append({
            "role": "assistant",
            "content": ranking,
        })

    messages.append({
        "role": "user",
        "content": (
            "این یک مورد جدید و مستقل است.\n\n"
            f"{query}\n{candidate_block}"
        ),
    })
    return messages


In [ ]:
if RUN_HEAVY_EXPERIMENTS:
    load_quantized_model(
        DORNA_MODEL_NAME,
        trust_remote_code=True,
    )

    dorna_fair_results = [
        evaluate_psc_checkpointed(
            experiment_tag=DORNA_FAIR_TAG,
            strategy=strategy,
            max_instances=50,
            k=5,
            build_messages_fn=(
                build_dorna_fair_messages
            ),
            generate_fn=generate_batch,
        )
        for strategy in [
            "random",
            "domain",
            "domain_lexical",
        ]
    ]
    display(pd.DataFrame(
        dorna_fair_results
    ))


### Recorded Dorna2 fair-condition results

The following values were obtained from the completed fair-condition run using the same lean listwise prompt structure, PSC with **k=5**, Borda aggregation, and **n=50 per strategy**.

| Strategy | Hits@1 | MRR | Winner consistency |
|---|---:|---:|---:|
| random | **0.78** | **0.8767** | **0.46** |
| domain | **0.68** | **0.8007** | **0.36** |
| domain_lexical | **0.50** | **0.6693** | **0.18** |

**Interpretation.** Dorna2 performs strongly on the easier random split, declines on domain negatives, and reaches Hits@1 = 0.50 on the hardest domain-lexical split. Under fair listwise PSC conditions, this is substantially stronger than the earlier exploratory Dorna2 pointwise and single-pass listwise results, but it remains slightly below Gemma's lean-prompt result of 0.52 on domain-lexical.

Expected detailed result files:

- `dorna2_fair_lean_psc_k5_random_summary.csv`
- `dorna2_fair_lean_psc_k5_domain_summary.csv`
- `dorna2_fair_lean_psc_k5_domain_lexical_summary.csv`


# 14. Unified recorded results

`status` distinguishes values embedded in the source files from local CSV results that still need to be copied into the shared Drive folder.


In [ ]:
RECORDED_RESULTS = [
    {
        "method": "TF-IDF / BM25",
        "protocol": "external PyCharm baseline",
        "random": None,
        "domain": None,
        "domain_lexical": 0.52,
        "n": 330,
        "status": "external code; result embedded",
    },
    {
        "method": "BGE-m3",
        "protocol": "external frozen dense baseline",
        "random": 0.894,
        "domain": None,
        "domain_lexical": 0.518,
        "n": 330,
        "status": "external code; result embedded",
    },
    {
        "method": "Gemma zero-shot",
        "protocol": "single-pass original order",
        "random": None,
        "domain": None,
        "domain_lexical": 0.491,
        "n": 330,
        "status": "embedded",
    },
    {
        "method": "Gemma zero-shot shuffled",
        "protocol": "non-aggregated, k=3",
        "random": None,
        "domain": None,
        "domain_lexical": 0.3667,
        "n": "50 × 3 generations",
        "status": "winner consistency 18%",
    },
    {
        "method": "Gemma lean prompt",
        "protocol": "PSC/Borda, k=10",
        "random": 0.96,
        "domain": 0.88,
        "domain_lexical": 0.52,
        "n": 50,
        "status": "embedded",
    },
    {
        "method": "Gemma CoP",
        "protocol": "PSC/Borda, k=5",
        "random": 0.96,
        "domain": 0.86,
        "domain_lexical": 0.50,
        "n": 50,
        "status": "domain_lexical approximate",
    },
    {
        "method": "Curriculum SFT",
        "protocol": "PSC/Borda, k=5",
        "random": None,
        "domain": None,
        "domain_lexical": 0.48,
        "n": 50,
        "status": "100% output validity",
    },
    {
        "method": "RAG-ICL",
        "protocol": "retrieval k=3; PSC k=5",
        "random": 0.94,
        "domain": 0.88,
        "domain_lexical": 0.52,
        "n": 50,
        "status": "embedded",
    },
    {
        "method": "Merge: RAG-augmented SFT",
        "protocol": "retrieval k=2; PSC k=5",
        "random": 0.94,
        "domain": 0.827,
        "domain_lexical": 0.594,
        "n": "50 / 330 / 330",
        "status": "embedded",
    },
    {
        "method": "Dorna2 pointwise",
        "protocol": "single-pass exploratory",
        "random": None,
        "domain": None,
        "domain_lexical": 0.3121,
        "n": 330,
        "status": "not fair-comparable",
    },
    {
        "method": "Dorna2 listwise",
        "protocol": "single-pass exploratory",
        "random": None,
        "domain": None,
        "domain_lexical": 0.2939,
        "n": 330,
        "status": "not fair-comparable",
    },
    {
        "method": "Llama-3.1-8B-Instruct",
        "protocol": "single-pass subset-50",
        "random": None,
        "domain": None,
        "domain_lexical": None,
        "n": 50,
        "status": "local summary CSV not embedded",
    },
    {
        "method": "Dorna2 fair conditions",
        "protocol": "lean listwise PSC k=5",
        "random": 0.78,
        "domain": 0.68,
        "domain_lexical": 0.50,
        "n": 50,
        "status": (
            "MRR random/domain/domain_lexical = "
            "0.8767 / 0.8007 / 0.6693; "
            "winner consistency = 0.46 / 0.36 / 0.18"
        ),
    },
]

results_table = pd.DataFrame(
    RECORDED_RESULTS
)
results_table


### Static comparison table — visible without executing the notebook

The table below records the principal **Hits@1** results available from the completed experiments. A dash means that the corresponding result was not contained in the supplied source files; it does not mean zero.

| Method | Protocol | Random | Domain | Domain-lexical | Evaluation size |
|---|---|---:|---:|---:|---:|
| TF-IDF / BM25 | external PyCharm baseline | — | — | **0.520** | 330 |
| BGE-m3 | frozen dense baseline | **0.894** | — | **0.518** | 330 |
| Gemma zero-shot | original-order single pass | — | — | **0.491** | 330 |
| Gemma zero-shot shuffled | non-aggregated, k=3 | — | — | **0.3667** | 50 × 3 |
| Gemma lean prompt | PSC/Borda, k=10 | **0.960** | **0.880** | **0.520** | 50 each |
| Gemma Chain-of-Prompting | PSC/Borda, k=5 | **0.960** | **0.860** | **0.500** | 50 each |
| Curriculum SFT | PSC/Borda, k=5 | — | — | **0.480** | 50 |
| RAG-ICL | retrieval k=3; PSC k=5 | **0.940** | **0.880** | **0.520** | 50 each |
| Merge: RAG-augmented SFT | retrieval k=2; PSC k=5 | **0.940** | **0.827** | **0.594** | 50 / 330 / 330 |
| Dorna2 pointwise | exploratory single pass | — | — | **0.3121** | 330 |
| Dorna2 listwise | exploratory single pass | — | — | **0.2939** | 330 |
| Llama-3.1-8B-Instruct | single-pass subset | — | — | — | 50 each |
| **Dorna2 fair conditions** | **lean listwise PSC, k=5** | **0.780** | **0.680** | **0.500** | **50 each** |

For Dorna2 fair conditions, the accompanying MRR values are **0.8767**, **0.8007**, and **0.6693**, while winner consistency is **0.46**, **0.36**, and **0.18** for random, domain, and domain-lexical respectively.


In [ ]:
def import_summary_values(summary_files):
    frames = []

    for file in summary_files:
        file = Path(file)
        if not file.exists():
            print("Missing:", file)
            continue

        dataframe = pd.read_csv(file)
        dataframe["source_file"] = file.name
        frames.append(dataframe)

    if not frames:
        return pd.DataFrame()

    return pd.concat(
        frames,
        ignore_index=True,
    )

# After copying downloaded summaries into RESULTS_DIR:
# imported_summaries = import_summary_values(
#     RESULTS_DIR.glob("*_summary.csv")
# )
# display(imported_summaries)


# 15. Qualitative error inspection

The cleaned evaluators retain query text, every candidate, gold index, aggregated ranking, per-shuffle rankings, raw outputs, parse validity, stability, and elapsed time.


In [ ]:
def load_experiment_details(
    detail_jsonl,
):
    return pd.DataFrame(
        load_jsonl(detail_jsonl)
    )

def show_errors(
    detail_jsonl,
    max_rows=10,
):
    dataframe = load_experiment_details(
        detail_jsonl
    )
    errors = dataframe[
        dataframe["gold_rank"] > 1
    ].copy()

    preferred_columns = [
        "strategy",
        "instance_index",
        "query",
        "candidate_texts",
        "gold_idx",
        "gold_rank",
        "aggregated_ranking",
        "per_shuffle_rankings",
        "winner_consistent",
        "raw_outputs",
    ]
    available_columns = [
        column
        for column in preferred_columns
        if column in errors.columns
    ]
    return errors[
        available_columns
    ].head(max_rows)

# Example:
# show_errors(
#     RESULTS_DIR /
#     "gemma2_9b_it_rag_augmented_sft_psc_k5_"
#     "domain_lexical_details.jsonl"
# )


# 16. Main conclusions

1. Original-order single-pass LLM evaluation was materially affected by candidate position.
2. The heavy prompt produced a fixed-ranking anchoring failure.
3. The lean prompt removed that collapse and established the strongest prompt-only baseline.
4. CoP increased inference cost without adding ranking signal.
5. Curriculum SFT improved output reliability but not hard-split Hits@1.
6. RAG-ICL improved winner consistency but not top-1 accuracy.
7. Merge was the only tested method to exceed the approximately 0.52 domain-lexical ceiling, reaching 0.594 at n=330.
8. Under the fair listwise PSC protocol, Dorna2 reached Hits@1 values of **0.78 random**, **0.68 domain**, and **0.50 domain-lexical**. This substantially improves on its earlier exploratory protocols, but remains below Gemma on all three matched subset-50 comparisons.
9. The Dorna2 winner-consistency results—0.46 random, 0.36 domain, and 0.18 domain-lexical—show that ranking stability also declines as negative candidates become harder.
